# RAG Chatbot - Legal/Professional Version

## What This Does

This notebook creates a chatbot that answers questions about your PDF documents using **HuggingFace Inference API**.

**Designed for:**
- Legal document Q&A
- Professional knowledge bases
- Research document analysis
- Any document-based Q&A application

**You Need:**
- PDF files in your Google Drive
- A free HuggingFace API token (~300 requests/hour)
- Internet connection

**Cost:** 100% FREE

**Features:**
- Consolidated configuration (all settings in one place)
- Source citations (see which documents were used)
- Conversation memory (follow-up questions work)
- 30-second timeout protection

---

**Simplified from:** RAG_Chatbot_HuggingFace.ipynb (empathy features removed)

---
## Step 1: Install Libraries

This installs the required tools. Takes about 30 seconds.

In [ ]:
# Install all required packages (no empathy/visualization packages)
!pip install -q chromadb gradio pypdf sentence-transformers huggingface_hub

print("✅ All libraries installed successfully!")
print("\nℹ️  Note: You may see dependency warnings about 'opentelemetry' packages.")
print("   These are non-critical and won't affect functionality. You can safely ignore them.")

---
## Step 2: Load Libraries

Load the tools we just installed.

In [ ]:
import os
import time
import asyncio
import gradio as gr
from huggingface_hub import InferenceClient
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

---
## Step 3: Connect Google Drive

**Steps:**
1. Click the link that appears
2. Choose your Google account
3. Click "Allow"

Your files will be at: `/content/drive/MyDrive/`

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")
print("📁 Your files are available at: /content/drive/MyDrive/")

---
## Step 4: Configuration - ALL SETTINGS HERE! ✏️

**⚠️ IMPORTANT: All configurable options are consolidated below**

### Get Your FREE HuggingFace Token:
1. Go to: https://huggingface.co/
2. Sign up for free account (email + password)
3. Go to: https://huggingface.co/settings/tokens
4. Click "New token"
5. Name it: "RAG Chatbot"
6. Type: "Fine-grained" with "Make calls to Inference Providers" enabled
7. Click "Generate"
8. Copy the token (starts with `hf_`)
9. Paste it below

**Free Tier:** ~300 requests per hour

In [ ]:
# ============================================================================
#                    RAG CHATBOT - CONFIGURATION
# ============================================================================
#
# All settings are organized into sections below.
# Required settings are marked with ✏️
# Optional settings have sensible defaults.
#
# ============================================================================

# ============================================================================
# SECTION 1: API CREDENTIALS (REQUIRED) ✏️
# ============================================================================
# Get your FREE token from: https://huggingface.co/settings/tokens
# Create a "Fine-grained" token with "Make calls to Inference Providers" enabled

HUGGINGFACE_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"  # ✏️ REPLACE THIS (starts with hf_)

# ============================================================================
# SECTION 2: MODEL SETTINGS
# ============================================================================

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # ✅ Recommended

# Alternative models (uncomment to try):
# MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"  # Fastest, most reliable
# MODEL_NAME = "google/gemma-2-2b-it"  # Lightweight, instruction-tuned

TEMPERATURE = 0.7       # Creativity (0.0 = focused, 1.0 = creative)
MAX_OUTPUT_TOKENS = 300 # Maximum response length (~225 words)

# ============================================================================
# SECTION 3: PERSONA CONFIGURATION (REQUIRED) ✏️
# ============================================================================

PERSONA_NAME = "Legal Expert"  # ✏️ CHANGE THIS - e.g., "Contract Analyst", "Research Assistant"

# ✏️ CUSTOMIZE THIS: Describe how the assistant should communicate
PERSONA_DESCRIPTION = """
You are a knowledgeable legal assistant specializing in document analysis.
You speak in a clear, professional manner, using precise legal terminology when appropriate.
You cite specific sections and clauses from documents to support your answers.
You acknowledge uncertainty when information is incomplete or ambiguous.

Instructions for customization:
- Describe the communication style (formal, technical, conversational)
- Specify domain expertise (contracts, regulations, research)
- Note any specific requirements (cite sources, highlight risks, etc.)
"""

# ============================================================================
# SECTION 4: PDF DOCUMENTS (REQUIRED) ✏️
# ============================================================================
# Format: "/content/drive/MyDrive/folder_name/file_name.pdf"
# TIP: Right-click a file in Colab's file browser → "Copy path"

PDF_PATHS = [
    "/content/drive/MyDrive/your_folder/document1.pdf",  # ✏️ REPLACE
    "/content/drive/MyDrive/your_folder/document2.pdf",  # ✏️ Add more as needed
    "/content/drive/MyDrive/your_folder/document3.pdf",
    # Add more PDFs here...
]

# ============================================================================
# SECTION 5: RETRIEVAL SETTINGS
# ============================================================================

NUM_RETRIEVED_DOCS = 7  # How many document pieces to search (then use top 3)
CHUNK_SIZE = 1000       # Characters per chunk (500-2000 recommended)
OVERLAP = 200           # Overlap between chunks (prevents mid-sentence splits)

# ============================================================================
# SECTION 6: CONVERSATION SETTINGS
# ============================================================================

CONVERSATION_MEMORY = 3  # How many previous exchanges to remember (0 = no memory)
                         # Higher values = more context but slower responses

SHOW_SOURCES = True      # True = show which PDFs were used, False = hide

DEBUG_MEMORY = False     # Set to True to see conversation history sent to API

# ============================================================================
# SECTION 7: STARTER QUESTIONS (OPTIONAL)
# ============================================================================
# These appear as clickable examples when chat starts

STARTER_QUESTIONS = [
    "What are the key terms in this document?",
    "Summarize the main obligations and responsibilities.",
    "Are there any potential risks or concerns?",
    "What are the termination conditions?",
    "Explain the liability clauses.",
]

# ============================================================================
# INITIALIZATION (Don't change this part)
# ============================================================================
client = InferenceClient(token=HUGGINGFACE_TOKEN)

print("✅ Configuration complete!")
print("="*60)
print(f"📋 Persona: {PERSONA_NAME}")
print(f"🤖 Model: {MODEL_NAME}")
print(f"📄 PDF files: {len(PDF_PATHS)}")
print(f"🌡️  Temperature: {TEMPERATURE}")
print(f"📊 Retrieved docs: {NUM_RETRIEVED_DOCS}")
print(f"📏 Chunk size: {CHUNK_SIZE} chars (overlap: {OVERLAP})")
print(f"🧠 Conversation memory: {CONVERSATION_MEMORY} exchanges")
print(f"📚 Show sources: {'ON ✅' if SHOW_SOURCES else 'OFF'}")
print(f"🧪 Debug mode: {'ON 🔍' if DEBUG_MEMORY else 'OFF'}")
print("="*60)

---
## Step 5: Test API Connection

**Run this first!** This checks if your API token works.

✅ If successful: Continue to next step  
❌ If failed: Check your API token and try again

In [ ]:
print("🧪 Testing HuggingFace API connection...")
print("=" * 60)

try:
    # Test using chat_completion API
    test_response = client.chat_completion(
        messages=[
            {"role": "system", "content": PERSONA_DESCRIPTION},
            {"role": "user", "content": "Say 'Hello! API is working!' in a professional style."}
        ],
        model=MODEL_NAME,
        max_tokens=50,
        temperature=0.7
    )
    
    response_text = test_response.choices[0].message.content
    
    print("✅ SUCCESS! HuggingFace API is working!")
    print(f"\nTest Response: {response_text}")
    print("\n" + "=" * 60)
    print("✅ You can proceed with the rest of the notebook!")
    
except Exception as e:
    error_str = str(e).lower()
    print(f"❌ API TEST FAILED!")
    print(f"Error: {str(e)}")
    print("\n" + "=" * 60)
    print("⚠️  STOP! Fix this issue before proceeding:")
    
    if "503" in str(e) or "loading" in error_str:
        print("  🔄 Model is loading (can take 20-30 seconds)")
        print("  💡 SOLUTION: Wait 30 seconds and run this cell again")
    elif "invalid" in error_str or "token" in error_str or "401" in str(e) or "403" in str(e):
        print("  🔑 Token issue detected")
        print("  💡 SOLUTION:")
        print("     1. Go to https://huggingface.co/settings/tokens")
        print("     2. Create 'Fine-grained' token")
        print("     3. Enable 'Make calls to Inference Providers'")
        print("     4. Copy new token to Step 4 configuration")
    elif "model" in error_str and ("not found" in error_str or "does not exist" in error_str):
        print("  🤖 Model not available")
        print("  💡 SOLUTION: Try 'microsoft/Phi-3.5-mini-instruct' in Step 4")
    else:
        print("  1. Check your API token is correct (starts with hf_)")
        print("  2. Check your internet connection")
        print("  3. Make sure token has 'Inference' permissions")

---
## Step 6: Read PDF Files

This reads your PDFs and splits them into small pieces for searching.

**Time:** 1-2 minutes depending on file size

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Get text from a PDF file."""
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    except Exception as e:
        print(f"❌ Error reading {pdf_path}: {str(e)}")
        return ""

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Split text into small pieces (chunks) for better searching."""
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        
        if chunk.strip():
            chunks.append(chunk)
        
        start += chunk_size - overlap
    
    return chunks

# Process all PDFs
print("📖 Reading PDF files...\n")
all_chunks = []
metadata = []

for idx, pdf_path in enumerate(PDF_PATHS):
    print(f"Processing: {pdf_path}")
    
    if not os.path.exists(pdf_path):
        print(f"⚠️  File not found - {pdf_path}")
        continue
    
    text = extract_text_from_pdf(pdf_path)
    
    if text:
        chunks = chunk_text(text)
        all_chunks.extend(chunks)
        
        for chunk_idx, chunk in enumerate(chunks):
            metadata.append({
                "source": os.path.basename(pdf_path),
                "chunk_id": chunk_idx,
                "total_chunks": len(chunks)
            })
        
        print(f"  ✅ Created {len(chunks)} pieces")
    else:
        print(f"  ⚠️  No text found")

print(f"\n✅ Done!")
print(f"📊 Total pieces: {len(all_chunks)}")
print(f"📏 Using chunk size: {CHUNK_SIZE} chars with {OVERLAP} char overlap")

if len(all_chunks) == 0:
    print("\n⚠️  WARNING: No text found in PDFs!")
    print("Check: File paths correct? PDFs not password-protected?")

---
## Step 7: Create Search Database

This creates a searchable database from your PDFs.

**Time:** 1-2 minutes

In [ ]:
# ============================================================================
# CREATE VECTOR DATABASE
# ============================================================================

print("🔧 Creating vector database...")
print("=" * 60)

# Step 1: Initialize embedding model
print("\n1️⃣ Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("   ✅ Embedding model loaded: all-MiniLM-L6-v2")

# Step 2: Create embeddings for all chunks
print("\n2️⃣ Creating embeddings for document chunks...")
print(f"   Processing {len(all_chunks)} chunks...")
embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)
print(f"   ✅ Created {len(embeddings)} embeddings")

# Step 3: Initialize ChromaDB
print("\n3️⃣ Initializing ChromaDB...")
chroma_client = chromadb.Client(Settings(
    anonymized_telemetry=False,
    allow_reset=True
))
print("   ✅ ChromaDB client initialized")

# Step 4: Create collection
print("\n4️⃣ Creating document collection...")
collection_name = "legal_documents"

try:
    chroma_client.delete_collection(name=collection_name)
    print("   🗑️  Cleared old collection")
except:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"description": "PDF document chunks for RAG chatbot"}
)
print(f"   ✅ Collection '{collection_name}' created")

# Step 5: Add documents to collection
print("\n5️⃣ Adding documents to database...")
print(f"   Uploading {len(all_chunks)} chunks...")

batch_size = 100
for i in range(0, len(all_chunks), batch_size):
    batch_end = min(i + batch_size, len(all_chunks))
    
    collection.add(
        documents=all_chunks[i:batch_end],
        embeddings=embeddings[i:batch_end].tolist(),
        metadatas=metadata[i:batch_end],
        ids=[f"chunk_{j}" for j in range(i, batch_end)]
    )
    
    print(f"   📦 Uploaded chunks {i+1}-{batch_end}/{len(all_chunks)}")

print("\n" + "=" * 60)
print("✅ Vector database created successfully!")
print(f"📊 Total chunks in database: {collection.count()}")
print(f"🔍 Database ready for semantic search!")
print("=" * 60)

---
## Step 8: Setup Question Answering

This prepares the chatbot to answer your questions.

In [ ]:
# Store which PDFs were used for the last answer
last_sources_used = []

def retrieve_relevant_context(query, n_results=NUM_RETRIEVED_DOCS):
    """Find relevant pieces from your PDFs based on the question."""
    global last_sources_used
    try:
        query_embedding = embedding_model.encode([query])
        
        results = collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=min(n_results, collection.count())
        )
        
        documents = results['documents'][0] if results['documents'] else []
        metadatas = results['metadatas'][0] if results['metadatas'] else []
        
        # Track which PDFs were used
        last_sources_used = []
        if metadatas:
            seen_sources = set()
            for meta in metadatas[:3]:
                source_name = meta.get('source', 'Unknown')
                if source_name not in seen_sources:
                    last_sources_used.append(source_name)
                    seen_sources.add(source_name)
        
        return documents
    except Exception as e:
        print(f"Error searching: {str(e)}")
        last_sources_used = []
        return []

def generate_response_sync(question, chat_history=None):
    """Get answer from AI using relevant PDF pieces + conversation memory."""
    # Find relevant pieces from PDFs
    context_docs = retrieve_relevant_context(question)
    
    if context_docs:
        context_docs = context_docs[:3]  # Use top 3 most relevant chunks
        context = "\n\n".join(context_docs)
        context = context[:3000]  # Limit to 3000 characters
    else:
        context = "No relevant documents found."
    
    # Build messages with system prompt + context
    messages = [
        {
            "role": "system",
            "content": f"""{PERSONA_DESCRIPTION}

ANSWERING INSTRUCTIONS:
Use the information below from the documents to answer the question.

CONTEXT FROM DOCUMENTS:
{context}

GUIDELINES:
- Ground your answer in the specific facts and details from the context above
- If the context doesn't fully answer the question, acknowledge what you know and what you don't
- Cite specific sections when relevant
- Be clear and professional"""
        }
    ]
    
    # Add conversation history for context
    if chat_history and CONVERSATION_MEMORY > 0:
        recent_history = chat_history[-(CONVERSATION_MEMORY):]
        for user_msg, bot_msg in recent_history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": bot_msg})
    
    # Add current question
    messages.append({"role": "user", "content": question})
    
    # Debug output
    if DEBUG_MEMORY:
        print("\n" + "="*80)
        print("🧠 DEBUG: CONVERSATION MEMORY")
        print("="*80)
        print(f"📊 Sending {len(messages)} messages to API:")
        for i, msg in enumerate(messages):
            role_emoji = "🖥️" if msg['role'] == "system" else ("👤" if msg['role'] == "user" else "🤖")
            content_preview = msg['content'][:60].replace('\n', ' ').strip()
            print(f"   [{i}] {role_emoji} {msg['role']:10} | {content_preview}...")
        print("="*80 + "\n")
    
    try:
        response = client.chat_completion(
            messages=messages,
            model=MODEL_NAME,
            max_tokens=MAX_OUTPUT_TOKENS,
            temperature=TEMPERATURE
        )
        
        return response.choices[0].message.content
        
    except Exception as e:
        error_str = str(e).lower()
        
        if "503" in str(e) or "loading" in error_str:
            return "⏳ Model is loading... Please wait 20-30 seconds and try again."
        elif "model" in error_str and "not found" in error_str:
            return "❌ Model not available. Try 'microsoft/Phi-3.5-mini-instruct' in Step 4"
        elif "429" in str(e) or "rate limit" in error_str:
            return "⚠️ Rate limit reached. Wait 10-15 minutes (free tier: ~300 requests/hour)"
        else:
            raise e

async def generate_response_async(question, chat_history=None, timeout_seconds=30):
    """Wrapper with 30-second timeout to prevent hanging."""
    try:
        response_text = await asyncio.wait_for(
            asyncio.to_thread(generate_response_sync, question, chat_history),
            timeout=timeout_seconds
        )
        return response_text
    
    except asyncio.TimeoutError:
        return "⏱️ **Timeout** - Took too long (>30 seconds). Try a simpler question."
    
    except Exception as e:
        error_str = str(e).lower()
        
        if "429" in str(e) or "quota" in error_str or "rate limit" in error_str:
            return "⚠️ **Rate Limit** - Wait 10-15 minutes and try again."
        elif "timeout" in error_str or "connection" in error_str:
            return "⚠️ **Connection Error** - Check your internet."
        elif "invalid" in error_str or "token" in error_str or "401" in str(e):
            return "⚠️ **API Error** - Check your HuggingFace token permissions."
        elif "503" in str(e) or "loading" in error_str:
            return "⚠️ **Model Loading** - Wait 20-30 seconds and try again."
        else:
            return f"❌ **Error** - {str(e)[:100]}"

print("✅ Question answering system ready!")
print(f"🤖 Model: {MODEL_NAME}")
print(f"🧠 Conversation memory: {'ENABLED (' + str(CONVERSATION_MEMORY) + ' exchanges)' if CONVERSATION_MEMORY > 0 else 'DISABLED'}")
print(f"🧪 Debug mode: {'ENABLED' if DEBUG_MEMORY else 'DISABLED'}")
print("📏 Context: 3 chunks, 3000 chars max")
print(f"📝 Max output: {MAX_OUTPUT_TOKENS} tokens")
print("⏱️  Response time: 5-15 seconds")
if SHOW_SOURCES:
    print("📚 Source citations enabled")

---
## Step 9: Launch Chat Interface

This creates the web-based chat interface using Gradio.

**What happens when you run this:**
- Gradio creates a public web link (works for 72 hours)
- Your chatbot goes live
- You can share the link with others

**⚠️ IMPORTANT:** 
- Copy the `https://xxxxx.gradio.live` link
- Open it in a NEW browser tab (not in Colab)
- The interface in Colab can be unstable

In [ ]:
async def chat_interface(message, history):
    """Handle chat messages with conversation memory and source citations."""
    # Get answer from AI (with conversation memory if enabled)
    response = await generate_response_async(message, history)
    
    # Add source citations if enabled
    if SHOW_SOURCES and last_sources_used:
        response += "\n\n---\n**📚 Sources:**\n"
        for i, source in enumerate(last_sources_used, 1):
            response += f"{i}. {source}\n"
    
    return response

# Create chat interface
memory_status = f"🧠 Conversation memory: {'ENABLED (' + str(CONVERSATION_MEMORY) + ' exchanges)' if CONVERSATION_MEMORY > 0 else 'DISABLED'}"

demo = gr.ChatInterface(
    fn=chat_interface,
    title=f"🤖 {PERSONA_NAME} - Document Q&A",
    description=f"""Ask questions about your documents.
    
    {memory_status}
    {'📖 Source citations enabled - see which PDFs were used' if SHOW_SOURCES else ''}
    
    ⏱️ Response time: 5-15 seconds
    """,
    examples=STARTER_QUESTIONS,
)

# Launch chat
print("=" * 80)
print("🚀 LAUNCHING CHAT INTERFACE")
print("=" * 80)
print(f"\n📋 Persona: {PERSONA_NAME}")
print(f"🧠 Memory: {'ENABLED (' + str(CONVERSATION_MEMORY) + ' exchanges)' if CONVERSATION_MEMORY > 0 else 'DISABLED'}")
if SHOW_SOURCES:
    print("📚 Sources: ON")
print("\n⚠️  IMPORTANT: Use the PUBLIC LINK below (not Colab interface)\n")
print("👇 COPY THIS LINK AND OPEN IN NEW TAB:\n")

demo.launch(
    share=True,
    inline=False,
    debug=True
)

print("\n" + "=" * 80)
print("✅ Chat is live!")
print("=" * 80)
print("\n📌 STEPS:")
print("   1. Find 'Running on public URL:' above")
print("   2. Copy the https://xxxxx.gradio.live link")
print("   3. Open in new browser tab")
print("   4. Start asking questions!")
if CONVERSATION_MEMORY > 0:
    print(f"   5. Follow-up questions work (remembers last {CONVERSATION_MEMORY} exchanges)")
if SHOW_SOURCES:
    print(f"   {'6' if CONVERSATION_MEMORY > 0 else '5'}. Check bottom of answers for sources")
print("\n💡 Link expires after 72 hours of no use\n")